In [4]:
"""
============================================================================
HIDDEN MARKOV MODEL (HMM) POS TAGGER - COMPLETE IMPLEMENTATION
============================================================================
Հեղինակ: Copilot for SNarek889
Նպատակ: Part-of-Speech Tagging with Brute Force Decoding
============================================================================
"""

from collections import defaultdict
from itertools import product
import math
from typing import List, Tuple, Dict


class HMMPOSTagger:
    """
    Hidden Markov Model-based Part-of-Speech Tagger

    Այս դասը իրականացնում է HMM ալգորիթմ POS tagging-ի համար:

    Հիմնական բաղադրիչներ:
    ---------------------
    1. Initial Probabilities:  P(tag)
    2. Transition Probabilities: P(tag_i | tag_{i-1})
    3. Emission Probabilities: P(word | tag)

    Decoding մեթոդ:
    ---------------
    Brute Force - ստուգում է բոլոր հնարավոր tag sequences-ները
    """

    def __init__(self, smoothing: float = 1.0):
        """
        Ինիցիալիզացիա

        Args:
            smoothing: Laplace smoothing պարամետր (default=1.0)
                      Օգտագործվում է անծանոթ բառերի/անցումների համար
        """
        self.smoothing = smoothing

        # ============================================
        # ՀԱՎԱՆԱԿԱՆՈՒԹՅՈՒՆՆԵՐԻ ՊԱՀՊԱՆՈՒՄ
        # ============================================
        self.initial_probs: Dict[str, float] = defaultdict(float)
        self.transition_probs: Dict[Tuple[str, str], float] = defaultdict(float)
        self.emission_probs: Dict[Tuple[str, str], float] = defaultdict(float)

        # ============================================
        # ՔԱՆԱԿՆԵՐԻ ՀԱՇՎՈՒՄ (Counts)
        # ============================================
        self.initial_counts: Dict[str, int] = defaultdict(int)
        self.transition_counts: Dict[Tuple[str, str], int] = defaultdict(int)
        self.emission_counts: Dict[Tuple[str, str], int] = defaultdict(int)
        self.tag_counts: Dict[str, int] = defaultdict(int)

        # ============================================
        # VOCABULARY
        # ============================================
        self.all_tags = set()
        self.all_words = set()

        self.trained = False


    def train(self, training_data: List[List[Tuple[str, str]]]) -> None:
        """
        Ուսուցում training data-ի վրա

        Args:
            training_data: List of sentences
                          Յուրաքանչյուր sentence = [(word, tag), ...]

        Օրինակ:
            [
                [("The", "DET"), ("dog", "NOUN"), ("barks", "VERB")],
                [("A", "DET"), ("cat", "NOUN"), ("meows", "VERB")],
                ...
            ]

        Քայլեր:
        -------
        1. Հաշվել բոլոր counts-ները (initial, transition, emission)
        2. Հաշվարկել հավանականությունները smoothing-ով
        """

        print("=" * 80)
        print("HMM TRAINING STARTED")
        print("=" * 80)
        print(f"Number of training sentences: {len(training_data)}")

        # ============================================
        # ՔԱՅԼ 1: ՀԱՇՎԵԼ COUNTS-ԵՐԸ
        # ============================================
        for sentence in training_data:
            prev_tag = None

            for i, (word, tag) in enumerate(sentence):
                # Ավելացնել vocabulary-ին
                self.all_tags.add(tag)
                self.all_words.add(word)

                # ❶ Initial counts (առաջին բառի tag-ը)
                if i == 0:
                    self.initial_counts[tag] += 1

                # ❷ Transition counts (tag-ից tag անցում)
                if prev_tag is not None:
                    self.transition_counts[(prev_tag, tag)] += 1

                # ❸ Emission counts (tag-ից word արտանետում)
                self.emission_counts[(tag, word)] += 1

                # ❹ Tag counts (ընդհանուր)
                self.tag_counts[tag] += 1

                prev_tag = tag

        # ============================================
        # ՔԱՅԼ 2: ՀԱՇՎԱՐԿԵԼ ՀԱՎԱՆԱԿԱՆՈՒԹՅՈՒՆՆԵՐԸ
        # ============================================

        num_tags = len(self.all_tags)
        num_words = len(self.all_words)
        total_sentences = len(training_data)

        print(f"\nVocabulary Statistics:")
        print(f"  • Unique tags: {num_tags}")
        print(f"  • Unique words: {num_words}")
        print(f"  • Tags: {sorted(self.all_tags)}")

        # ❶ INITIAL PROBABILITIES
        # P(tag) = (count(tag) + α) / (total_sentences + α × |tags|)
        print(f"\n{'='*80}")
        print("COMPUTING INITIAL PROBABILITIES")
        print(f"{'='*80}")

        for tag in self.all_tags:
            count = self.initial_counts[tag]
            self.initial_probs[tag] = (
                (count + self.smoothing) /
                (total_sentences + self.smoothing * num_tags)
            )
            print(f"  P({tag:8}) = {self.initial_probs[tag]:.4f}  (count={count})")

        # ❷ TRANSITION PROBABILITIES
        # P(tag_j | tag_i) = (count(tag_i → tag_j) + α) / (count(tag_i) + α × |tags|)
        print(f"\n{'='*80}")
        print("COMPUTING TRANSITION PROBABILITIES (Sample)")
        print(f"{'='*80}")

        sample_count = 0
        for prev_tag in self.all_tags:
            for curr_tag in self.all_tags:
                count = self.transition_counts[(prev_tag, curr_tag)]
                prev_count = self.tag_counts[prev_tag]

                self.transition_probs[(prev_tag, curr_tag)] = (
                    (count + self.smoothing) /
                    (prev_count + self.smoothing * num_tags)
                )

                # Print only first few for brevity
                if sample_count < 5 and count > 0:
                    prob = self.transition_probs[(prev_tag, curr_tag)]
                    print(f"  P({curr_tag:8} | {prev_tag:8}) = {prob:.4f}  (count={count})")
                    sample_count += 1

        print(f"  ... (total {num_tags * num_tags} transition probabilities computed)")

        # ❸ EMISSION PROBABILITIES
        # P(word | tag) = (count(tag → word) + α) / (count(tag) + α × |words|)
        print(f"\n{'='*80}")
        print("COMPUTING EMISSION PROBABILITIES (Sample)")
        print(f"{'='*80}")

        sample_count = 0
        for tag in self.all_tags:
            for word in self.all_words:
                count = self.emission_counts[(tag, word)]
                tag_count = self.tag_counts[tag]

                self.emission_probs[(tag, word)] = (
                    (count + self.smoothing) /
                    (tag_count + self.smoothing * num_words)
                )

                # Print only first few for brevity
                if sample_count < 5 and count > 0:
                    prob = self.emission_probs[(tag, word)]
                    print(f"  P(\"{word:10}\" | {tag:8}) = {prob:.4f}  (count={count})")
                    sample_count += 1

        print(f"  ... (total {num_tags * num_words} emission probabilities computed)")

        self.trained = True

        print(f"\n{'='*80}")
        print("TRAINING COMPLETE!")
        print(f"{'='*80}\n")


    def calculate_sequence_probability(
        self,
        words: List[str],
        tags: List[str]
    ) -> float:
        """
        Հաշվարկել tag sequence-ի հավանականությունը

        P(tags, words) = P(tag₁) × P(word₁|tag₁) ×
                         P(tag₂|tag₁) × P(word₂|tag₂) × ...

        Args:
            words: Բառերի ցուցակ
            tags: Tag-երի ցուցակ (նույն երկարությամբ)

        Returns:
            Log հավանականություն (log space-ում underflow-ից խուսափելու համար)
        """

        if len(words) != len(tags):
            raise ValueError("Words և tags-երը պետք է նույն երկարությունն ունենան")

        log_prob = 0.0

        for i, (word, tag) in enumerate(zip(words, tags)):
            # ❶ Initial կամ Transition probability
            if i == 0:
                prob = self.initial_probs.get(
                    tag,
                    self.smoothing / (len(self.all_tags) * self.smoothing)
                )
            else:
                prev_tag = tags[i-1]
                prob = self.transition_probs.get(
                    (prev_tag, tag),
                    self.smoothing / (len(self.all_tags) * self.smoothing)
                )

            # ❷ Emission probability
            emission_prob = self.emission_probs.get(
                (tag, word),
                self.smoothing / (len(self.all_words) * self.smoothing)
            )

            # Log space-ում գումարում ենք (բազմապատկման փոխարեն)
            log_prob += math.log(prob) + math.log(emission_prob)

        return log_prob


    def brute_force_decode(self, sentence: List[str]) -> List[str]:
        """
        BRUTE FORCE DECODING

        Գտնել ամենալավ tag sequence-ը՝ ստուգելով ԲՈԼՈՐ հնարավորները

        Ալգորիթմ:
        ----------
        1. Գեներացնել բոլոր հնարավոր tag sequences-ները
           (Եթե T tags և N words → T^N sequences)
        2. Հաշվարկել յուրաքանչյուրի հավանականությունը
        3. Վերադարձնել ամենամեծը

        Time Complexity: O(T^N × N) - ՇԱՏ ԴԱՆԴԱՂ!

        Args:
            sentence: Բառերի ցուցակ

        Returns:
            Ամենալավ tag sequence-ը
        """

        if not self.trained:
            raise ValueError(" Model-ը պետք է ուսուցված լինի decode անելուց առաջ!")

        if isinstance(sentence, str):
            sentence = sentence.split()

        n = len(sentence)
        all_tags_list = list(self.all_tags)

        print(f"\n{'='*80}")
        print(f"BRUTE FORCE DECODING")
        print(f"{'='*80}")
        print(f"Sentence: {' '.join(sentence)}")
        print(f"Length: {n} words")
        print(f"Possible tags per word: {len(all_tags_list)}")
        print(f"Total sequences to check: {len(all_tags_list)}^{n} = {len(all_tags_list)**n:,}")

        # Զգուշացում երկար նախադասությունների համար
        if len(all_tags_list)**n > 100000:
            print(f"\n️  WARNING: This will take a VERY long time!")
            print(f"   Consider using Viterbi algorithm instead.")

        # Գեներացնել բոլոր հնարավոր sequences-ները
        all_possible_sequences = product(all_tags_list, repeat=n)

        best_sequence = None
        best_prob = float('-inf')

        sequences_checked = 0

        # Ստուգել յուրաքանչյուր sequence
        for tag_sequence in all_possible_sequences:
            sequences_checked += 1

            # Հաշվարկել հավանականությունը
            prob = self.calculate_sequence_probability(sentence, tag_sequence)

            # Թարմացնել լավագույնը
            if prob > best_prob:
                best_prob = prob
                best_sequence = tag_sequence

                # Print progress for long sequences
                if sequences_checked % 1000 == 0:
                    print(f"  Checked {sequences_checked:,} sequences... "
                          f"Current best prob: {best_prob:.2f}")

        print(f"\n Checked {sequences_checked:,} sequences")
        print(f" Best sequence probability (log): {best_prob:.4f}")
        print(f" Best sequence probability: {math.exp(best_prob):.2e}")
        print(f"{'='*80}\n")

        return list(best_sequence)


    def tag_sentence(self, sentence: List[str]) -> List[Tuple[str, str]]:
        """
        Tag a sentence և վերադարձնել (word, tag) pairs

        Args:
            sentence: Բառերի ցուցակ կամ string

        Returns:
            [(word, tag), ...] pairs
        """
        if isinstance(sentence, str):
            sentence = sentence.split()

        tags = self.brute_force_decode(sentence)
        return list(zip(sentence, tags))


# ============================================================================
# ՕՐԻՆԱԿ / TESTING
# ============================================================================

def main():
    """
    Main function - Ցույց տալ HMM POS Tagger-ի աշխատանքը
    """

    print("\n" + "="*80)
    print(" " * 20 + "HMM POS TAGGER - BRUTE FORCE")
    print("="*80 + "\n")

    # ============================================
    # TRAINING DATA
    # ============================================
    training_data = [
        [("The", "DET"), ("dog", "NOUN"), ("barks", "VERB")],
        [("The", "DET"), ("cat", "NOUN"), ("meows", "VERB")],
        [("A", "DET"), ("dog", "NOUN"), ("runs", "VERB")],
        [("The", "DET"), ("dog", "NOUN"), ("runs", "VERB")],
        [("A", "DET"), ("cat", "NOUN"), ("sleeps", "VERB")],
        [("The", "DET"), ("big", "ADJ"), ("dog", "NOUN"), ("barks", "VERB")],
        [("A", "DET"), ("small", "ADJ"), ("cat", "NOUN"), ("meows", "VERB")],
        [("The", "DET"), ("dog", "NOUN"), ("sleeps", "VERB")],
        [("The", "DET"), ("cat", "NOUN"), ("runs", "VERB")],
        [("A", "DET"), ("big", "ADJ"), ("dog", "NOUN"), ("sleeps", "VERB")],
        [("The", "DET"), ("small", "ADJ"), ("cat", "NOUN"), ("runs", "VERB")],
        [("Dogs", "NOUN"), ("bark", "VERB")],
        [("Cats", "NOUN"), ("meow", "VERB")],
        [("The", "DET"), ("dog", "NOUN"), ("in", "ADP"), ("the", "DET"),
         ("house", "NOUN"), ("barks", "VERB")],
        [("A", "DET"), ("cat", "NOUN"), ("on", "ADP"), ("the", "DET"),
         ("mat", "NOUN"), ("sleeps", "VERB")],
    ]

    # ============================================
    # CREATE AND TRAIN
    # ============================================
    tagger = HMMPOSTagger(smoothing=1.0)
    tagger.train(training_data)

    # ============================================
    # TEST SENTENCES
    # ============================================
    print("="*80)
    print(" " * 30 + "TESTING")
    print("="*80 + "\n")

    test_sentences = [
        ["The", "dog", "barks"],
        ["A", "cat", "sleeps"],
        ["The", "big", "dog", "runs"],
        ["Dogs", "bark"],
    ]

    for sentence in test_sentences:
        tagged = tagger.tag_sentence(sentence)

        print("Result:")
        for word, tag in tagged:
            print(f"  {word:12} → {tag}")
        print()

    # ============================================
    # COMPLEXITY ANALYSIS
    # ============================================
    print("="*80)
    print(" " * 25 + "COMPLEXITY ANALYSIS")
    print("="*80 + "\n")

    T = len(tagger.all_tags)

    print(" BRUTE FORCE:")
    print(f"   Time Complexity: O(T^N × N)")
    print(f"   Where T = {T} tags, N = sentence length\n")

    for n in [3, 4, 5, 6]:
        sequences = T ** n
        print(f"   N = {n} words → {T}^{n} = {sequences:,} sequences")

    print(f"\n   ⚠  Grows EXPONENTIALLY!")

    print(f"\n{'='*80}")
    print("VITERBI (Dynamic Programming):")
    print(f"   Time Complexity: O(T² × N)")
    print(f"   Where T = {T} tags, N = sentence length\n")

    for n in [3, 4, 5, 6, 10, 20]:
        operations = T * T * n
        print(f"   N = {n} words → {T}² × {n} = {operations:,} operations")

    print(f"\n    MUCH more efficient for longer sentences!")
    print("="*80 + "\n")


if __name__ == "__main__":
    main()


                    HMM POS TAGGER - BRUTE FORCE

HMM TRAINING STARTED
Number of training sentences: 15

Vocabulary Statistics:
  • Unique tags: 5
  • Unique words: 19
  • Tags: ['ADJ', 'ADP', 'DET', 'NOUN', 'VERB']

COMPUTING INITIAL PROBABILITIES
  P(DET     ) = 0.7000  (count=13)
  P(ADJ     ) = 0.0500  (count=0)
  P(NOUN    ) = 0.1500  (count=2)
  P(ADP     ) = 0.0500  (count=0)
  P(VERB    ) = 0.0500  (count=0)

COMPUTING TRANSITION PROBABILITIES (Sample)
  P(ADJ      | DET     ) = 0.2500  (count=4)
  P(NOUN     | DET     ) = 0.6000  (count=11)
  P(NOUN     | ADJ     ) = 0.5556  (count=4)
  P(ADP      | NOUN    ) = 0.1364  (count=2)
  P(VERB     | NOUN    ) = 0.7273  (count=15)
  ... (total 25 transition probabilities computed)

COMPUTING EMISSION PROBABILITIES (Sample)
  P("The       " | DET     ) = 0.2647  (count=8)
  P("A         " | DET     ) = 0.1765  (count=5)
  P("the       " | DET     ) = 0.0882  (count=2)
  P("big       " | ADJ     ) = 0.1304  (count=2)
  P("small     " 